# Mejorando YOLO con SAHI — Demo interactiva
## Detección y segmentación de objetos pequeños en vista aérea urbana
### FLISOL 2026 · Computer Vision Open Source

En este notebook vamos a:

1. Ver **por qué YOLO falla** con objetos pequeños en imágenes de alta resolución.
2. Aplicar **SAHI** (slicing) y observar cómo recupera esos objetos.
3. Experimentar en vivo con los **parámetros clave**: tamaño de slice, solapamiento y postprocesamiento (NMS / NMM / GREEDYNMM).
4. Extender la idea a **segmentación de instancias** (YOLO-seg + SAHI con máscaras).

Dataset: **VisDrone2019-DET** (imágenes de dron urbanas, cientos de objetos diminutos por imagen).

> ⚙️ **Activa la GPU**: `Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)`.

---
## 1. Instalación y entorno

- `ultralytics`: modelos YOLO11 (detección y segmentación).
- `sahi`: el slicing y la fusión de detecciones.
- `supervision` / `matplotlib`: visualización.

In [ ]:
!pip install -q ultralytics sahi supervision matplotlib

import torch, time, glob
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available(), '| device:', DEVICE)

---
## 2. El problema, en una imagen

Un detector como YOLO **redimensiona toda la imagen** a su tamaño de entrada (típicamente 640×640).

| Imagen original | Tras redimensionar a 640 | Efecto |
|---|---|---|
| 1920×1080 | 640×360 (escala 0.33) | una persona de 30px → 10px |
| 3840×2160 (4K) | 640×360 (escala 0.17) | una moto de 40px → 7px |

A 7-10 píxeles el objeto pierde casi toda su información y el detector **no lo ve**.
Vamos a comprobarlo con una imagen real de VisDrone.

### 2.1 Descargar VisDrone (val) y elegir una imagen densa

In [ ]:
import urllib.request, zipfile, os
URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip'
if not os.path.exists('VisDrone2019-DET-val'):
    print('Descargando VisDrone val (~80 MB)...')
    urllib.request.urlretrieve(URL, 'visdrone_val.zip')
    with zipfile.ZipFile('visdrone_val.zip') as z: z.extractall('.')
    print('Listo.')

imgs = sorted(glob.glob('VisDrone2019-DET-val/images/*.jpg'))
print('Imágenes disponibles:', len(imgs))

from PIL import Image
IMG = imgs[3]   # cambia el índice para probar otras escenas
im = Image.open(IMG)
print('Imagen:', IMG, '| resolución:', im.size)
im

---
## 3. YOLO tradicional (el *antes*)

Usamos `yolo11s.pt` (pre-entrenado en COCO). Detecta personas, autos, motos, etc.
Fíjate en cuántos objetos lejanos se **quedan sin caja**.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')

t0 = time.perf_counter()
res = model.predict(IMG, conf=0.25, device=DEVICE, verbose=False)[0]
dt_base = time.perf_counter() - t0
n_base = len(res.boxes)
print(f'YOLO baseline: {n_base} detecciones en {dt_base:.3f}s')

res.save('baseline.jpg')
Image.open('baseline.jpg')

---
## 4. YOLO + SAHI (el *después*)

SAHI **no reemplaza** a YOLO: lo envuelve. En vez de aplastar la imagen:

1. La corta en mosaicos (*slices*) con solapamiento.
2. Corre YOLO en cada slice (el objeto ocupa más píxeles → se detecta).
3. Reproyecta las cajas a coordenadas globales.
4. **Fusiona** las detecciones duplicadas (postprocesamiento).

El objeto de SAHI se construye una vez y se reutiliza.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction

sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics', model_path='yolo11s.pt',
    confidence_threshold=0.25, device=DEVICE,
)

t0 = time.perf_counter()
result = get_sliced_prediction(
    IMG, sahi_model,
    slice_height=640, slice_width=640,
    overlap_height_ratio=0.2, overlap_width_ratio=0.2,
    postprocess_type='GREEDYNMM',
    postprocess_match_metric='IOS',
    postprocess_match_threshold=0.5,
    verbose=0,
)
dt_sahi = time.perf_counter() - t0
n_sahi = len(result.object_prediction_list)
print(f'YOLO + SAHI: {n_sahi} detecciones en {dt_sahi:.3f}s  ({dt_sahi/max(dt_base,1e-6):.1f}x mas lento)')

result.export_visuals(export_dir='.', file_name='sahi')
Image.open('sahi.png')

### 4.1 Comparativa lado a lado

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(22, 9))
ax[0].imshow(Image.open('baseline.jpg')); ax[0].set_title(f'YOLO  -  {n_base} objetos', fontsize=18); ax[0].axis('off')
ax[1].imshow(Image.open('sahi.png'));     ax[1].set_title(f'YOLO + SAHI  -  {n_sahi} objetos', fontsize=18); ax[1].axis('off')
plt.tight_layout(); plt.show()
print(f'SAHI encontro {n_sahi - n_base} objetos adicionales (x{n_sahi/max(n_base,1):.1f}).')

---
## 5. Parámetro clave 1 — Tamaño de slice

Slice más **pequeño** → cada objeto ocupa una fracción mayor del slice → detectas objetos más diminutos.
Pero más slices = **más tiempo**. Este es el trade-off central de SAHI.

Observa cómo suben las detecciones (y el tiempo) al reducir el slice.

In [ ]:
sizes, dets, times = [320, 512, 640, 768, 1024], [], []
for s in sizes:
    t0 = time.perf_counter()
    r = get_sliced_prediction(IMG, sahi_model, slice_height=s, slice_width=s,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              postprocess_type='GREEDYNMM', verbose=0)
    dt = time.perf_counter() - t0
    dets.append(len(r.object_prediction_list)); times.append(dt)
    print(f'slice={s:4d}px -> {len(r.object_prediction_list):4d} detecciones en {dt:.2f}s')

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar([str(s) for s in sizes], dets, color='#2E8B57', alpha=0.85)
ax1.set_xlabel('Tamano de slice (px)'); ax1.set_ylabel('Detecciones', color='#2E8B57')
ax2 = ax1.twinx(); ax2.plot([str(s) for s in sizes], times, 'o-', color='#E4572E', lw=2.5)
ax2.set_ylabel('Tiempo (s)', color='#E4572E')
plt.title('Precision vs velocidad segun el tamano de slice'); plt.show()

---
## 6. Parámetro clave 2 — Solapamiento (overlap)

Sin solape, un objeto justo en el borde de un slice se **parte en dos** y ninguna mitad se detecta bien.
El solape garantiza que cada objeto aparezca completo en al menos un slice. Más solape = más slices redundantes = más lento.

In [ ]:
for ov in [0.0, 0.1, 0.2, 0.3]:
    t0 = time.perf_counter()
    r = get_sliced_prediction(IMG, sahi_model, slice_height=512, slice_width=512,
                              overlap_height_ratio=ov, overlap_width_ratio=ov,
                              postprocess_type='GREEDYNMM', verbose=0)
    print(f'overlap={ov:.1f} -> {len(r.object_prediction_list):4d} detecciones en {time.perf_counter()-t0:.2f}s')

---
## 7. Parámetro clave 3 — Postprocesamiento (fusión de duplicados)

Con solape, el **mismo objeto** aparece en varios slices. Hay que fusionar esos duplicados. SAHI ofrece tres estrategias:

| Tipo | Qué hace |
|---|---|
| **NMS** | Non-Maximum Suppression: si dos cajas se solapan mucho, **descarta** la de menor score. |
| **NMM** | Non-Maximum Merging: en vez de descartar, **fusiona** las cajas solapadas en una sola. |
| **GREEDYNMM** | Variante voraz de NMM (valor por defecto de SAHI). |

Además, la **métrica de match** puede ser `IOU` (solape clásico) o `IOS` (Intersection over Smaller, mejor cuando una caja queda contenida en otra al cortar).

In [ ]:
for pp in ['NMS', 'NMM', 'GREEDYNMM']:
    t0 = time.perf_counter()
    r = get_sliced_prediction(IMG, sahi_model, slice_height=640, slice_width=640,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              postprocess_type=pp, postprocess_match_metric='IOS',
                              postprocess_match_threshold=0.5, verbose=0)
    print(f'{pp:10s} -> {len(r.object_prediction_list):4d} detecciones tras fusion en {time.perf_counter()-t0:.2f}s')

---
## 8. Modos de inferencia de SAHI

SAHI permite combinar la pasada por slices con una pasada sobre la imagen completa:

- **Sliced** (`get_sliced_prediction`): solo slices → mejor para objetos pequeños.
- **Standard** (`get_prediction`): imagen completa → mejor para objetos grandes.
- **Combinado**: `get_sliced_prediction(..., perform_standard_pred=True)` corre ambos y fusiona → lo más robusto, lo más lento.

El modo combinado cubre objetos grandes Y pequeños a la vez.

In [ ]:
r_solo_slice = get_sliced_prediction(IMG, sahi_model, slice_height=640, slice_width=640,
                                     overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                                     perform_standard_pred=False, postprocess_type='GREEDYNMM', verbose=0)
r_combinado  = get_sliced_prediction(IMG, sahi_model, slice_height=640, slice_width=640,
                                     overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                                     perform_standard_pred=True, postprocess_type='GREEDYNMM', verbose=0)
print('Solo sliced :', len(r_solo_slice.object_prediction_list), 'detecciones')
print('Combinado   :', len(r_combinado.object_prediction_list), 'detecciones (slices + imagen completa)')

---
## 9. SAHI con SEGMENTACIÓN de instancias

SAHI no se limita a cajas: también funciona con modelos de **segmentación** (máscaras por instancia).
Cada slice produce máscaras a resolución nativa y luego se fusionan, así que los objetos pequeños obtienen
máscaras mucho más precisas — o directamente **aparecen** cuando antes se perdían.

Usamos `yolo11s-seg.pt` (segmentación, pre-entrenado en COCO).

> Nota: VisDrone es un dataset de *detección* (sin máscaras), por eso esta sección es **cualitativa**:
> mostramos las máscaras sobre la imagen, no medimos mAP de segmentación.

In [ ]:
seg_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics', model_path='yolo11s-seg.pt',
    confidence_threshold=0.25, device=DEVICE,
)

# Segmentacion estandar (imagen completa)
seg_std = get_prediction(IMG, seg_model)
seg_std.export_visuals(export_dir='.', file_name='seg_std')

# Segmentacion con SAHI (slicing) -> mascaras tambien para objetos pequenos
seg_sahi = get_sliced_prediction(IMG, seg_model, slice_height=640, slice_width=640,
                                 overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                                 postprocess_type='GREEDYNMM', verbose=0)
seg_sahi.export_visuals(export_dir='.', file_name='seg_sahi')

n_seg_std = len(seg_std.object_prediction_list)
n_seg_sahi = len(seg_sahi.object_prediction_list)
print(f'Segmentacion  ->  YOLO-seg: {n_seg_std} instancias  |  YOLO-seg + SAHI: {n_seg_sahi} instancias')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(22, 9))
ax[0].imshow(Image.open('seg_std.png'));  ax[0].set_title(f'YOLO-seg  -  {n_seg_std} mascaras', fontsize=18); ax[0].axis('off')
ax[1].imshow(Image.open('seg_sahi.png')); ax[1].set_title(f'YOLO-seg + SAHI  -  {n_seg_sahi} mascaras', fontsize=18); ax[1].axis('off')
plt.tight_layout(); plt.show()

### 9.1 ¿Qué cambió en las máscaras?

- Con SAHI aparecen **máscaras de instancias pequeñas** (autos/personas lejanos) que el modelo estándar no segmentaba.
- Las máscaras existentes suelen quedar **más ajustadas al contorno** porque se calcularon a mayor resolución relativa.
- Mismo costo conceptual que en detección: más slices → más tiempo.

---
## 10. ¿Y los números (mAP)?

Esta demo es **cualitativa** (conteo y visual). Para el **benchmark riguroso de mAP** — incluyendo
`AP_small`, comparación de postprocesamiento y análisis por clase — usa el otro notebook:
**`02_resultados_figuras.ipynb`**, que evalúa contra el ground truth con `pycocotools` y genera
todas las figuras para las slides.

> Recuerda: el mAP real requiere un modelo **entrenado en VisDrone** (`src/train_visdrone.py`),
> porque las clases del modelo deben coincidir con las del ground truth.

---
## Conclusiones

- YOLO **aplasta** la imagen → pierde objetos pequeños. SAHI la **rebana** y los recupera, **sin reentrenar**.
- Parámetros que controlas: **tamaño de slice**, **overlap**, **modo** (sliced/standard/combinado) y **postprocesamiento** (NMS/NMM/GREEDYNMM + IOU/IOS).
- Funciona igual para **detección y segmentación**.
- Costo: **mayor tiempo de inferencia** (~proporcional al nº de slices).
- 100% **open source**: Ultralytics YOLO + SAHI.

Repo: github.com/TU-USUARIO/yolo-sahi-flisol2026